# 02 — Feature Engineering & EDA

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Load the preprocessed parquet data
2. One-hot encode genres and title types
3. Engineer runtime buckets, decade, language, director/cast features
4. Analyze language bias and numVotes (instructor feedback)
5. Build the final feature matrix and save it

In [1]:
# ── Imports & Environment Setup ──
import os
import sys
import numpy as np

# Fix Windows PySpark: set HADOOP_HOME and Python paths
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# ── Imports ──
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.window import Window

BASE_DIR = r"e:\CUFE\Spring_25\Big Data\Project"
DATA_DIR = os.path.join(BASE_DIR, "Dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")


spark = (
    SparkSession.builder
    .appName("IMDb_FeatureEngineering")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready.")

Spark 4.1.1 ready.


In [3]:
# ── Load preprocessed data ──
df = spark.read.csv(
    os.path.join(OUTPUT_DIR, "parquet", "preprocessed", "data.csv"),
    header=True, inferSchema=True
)
print(f"Loaded {df.count():,} rows, {len(df.columns)} columns")
df.printSchema()


Loaded 186,102 rows, 25 columns
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: double (nullable = true)
 |-- runtimeMinutes: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- averageRating: double (nullable = true)
 |-- numVotes: integer (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- primaryLanguage: string (nullable = true)
 |-- numRegions: double (nullable = true)
 |-- numPrincipals: double (nullable = true)
 |-- numRoleTypes: double (nullable = true)
 |-- userRatingCount: double (nullable = true)
 |-- userRatingMean: double (nullable = true)
 |-- userRatingStd: double (nullable = true)
 |-- userRatingMin: double (nullable = true)
 |-- userRatingMax: double (nullable = true)
 |-- extremeRatin

## 1. Genre One-Hot Encoding

In [4]:
# ── Split genres and one-hot encode ──
# Genres are comma-separated (e.g., "Action,Comedy,Drama")
df = df.withColumn("genreArray", F.split(F.col("genres"), ","))

# Get all unique genres
all_genres = (
    df.select(F.explode("genreArray").alias("genre"))
    .distinct()
    .orderBy("genre")
    .collect()
)
genre_list = [row["genre"] for row in all_genres if row["genre"] is not None]
print(f"Found {len(genre_list)} unique genres: {genre_list}")

# Create binary column for each genre
for genre in genre_list:
    col_name = f"genre_{genre.replace('-', '_')}"
    df = df.withColumn(
        col_name,
        F.when(F.array_contains("genreArray", genre), 1).otherwise(0)
    )

print(f"Added {len(genre_list)} genre columns")
genre_cols = [f"genre_{g.replace('-','_')}" for g in genre_list]
df.select(genre_cols[:5]).show(5)

Found 27 unique genres: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Talk-Show', 'Thriller', 'War', 'Western']
Added 27 genre columns
+------------+---------------+---------------+---------------+------------+
|genre_Action|genre_Adventure|genre_Animation|genre_Biography|genre_Comedy|
+------------+---------------+---------------+---------------+------------+
|           0|              0|              0|              0|           0|
|           0|              0|              0|              0|           0|
|           0|              0|              0|              0|           1|
|           0|              0|              0|              0|           0|
|           0|              0|              0|              0|           0|
+------------+---------------+---------------+---

## 2. Title Type Encoding

In [5]:
# ── One-hot encode title types ──
title_types = [row["titleType"] for row in df.select("titleType").distinct().collect()]
print(f"Title types: {title_types}")

for tt in title_types:
    col_name = f"type_{tt}"
    df = df.withColumn(col_name, F.when(F.col("titleType") == tt, 1).otherwise(0))

type_cols = [f"type_{tt}" for tt in title_types]
print(f"Added {len(type_cols)} title type columns")

Title types: ['movie', 'tvSeries', 'tvMovie', 'tvMiniSeries']
Added 4 title type columns


## 3. Runtime Buckets & Decade

In [6]:
# ── Runtime buckets ──
df = df.withColumn(
    "runtimeBucket",
    F.when(F.col("runtimeMinutes") < 90, "short")
    .when(F.col("runtimeMinutes") < 120, "standard")
    .when(F.col("runtimeMinutes") < 150, "long")
    .otherwise("very_long")
)

# One-hot encode runtime buckets
for bucket in ["short", "standard", "long", "very_long"]:
    df = df.withColumn(f"runtime_{bucket}", F.when(F.col("runtimeBucket") == bucket, 1).otherwise(0))

print("Runtime bucket distribution:")
df.groupBy("runtimeBucket").count().orderBy("count", ascending=False).show()

Runtime bucket distribution:
+-------------+-----+
|runtimeBucket|count|
+-------------+-----+
|     standard|83485|
|        short|78474|
|         long|17050|
|    very_long| 7093|
+-------------+-----+



In [7]:
# ── Decade feature ──
df = df.withColumn("decade", (F.floor(F.col("startYear") / 10) * 10).cast(IntegerType()))

print("Decade distribution:")
df.groupBy("decade").count().orderBy("decade").show(20)

Decade distribution:
+------+-----+
|decade|count|
+------+-----+
|  1890|    2|
|  1900|    5|
|  1910|  311|
|  1920|  941|
|  1930| 3773|
|  1940| 4013|
|  1950| 5675|
|  1960| 7598|
|  1970|10643|
|  1980|13062|
|  1990|17796|
|  2000|31387|
|  2010|54695|
|  2020|36201|
+------+-----+



## 4. Language Features (Bias Analysis per Instructor Feedback)

In [8]:
# ── Language bias analysis ──
# The instructor warned about skew towards certain features like language

print("=== Language Distribution (Top 20) ===")
lang_dist = df.groupBy("primaryLanguage").agg(
    F.count("*").alias("count"),
    F.round(F.mean("averageRating"), 2).alias("avgRating"),
    F.round(F.mean("numVotes"), 0).alias("avgVotes")
).orderBy("count", ascending=False)

lang_dist.show(20)

# Create isEnglish binary feature
df = df.withColumn("isEnglish", F.when(F.col("primaryLanguage") == "en", 1).otherwise(0))

eng_count = df.filter(F.col("isEnglish") == 1).count()
total = df.count()
print(f"\n⚠️ LANGUAGE BIAS: English titles = {eng_count:,}/{total:,} ({100*eng_count/total:.1f}%)")
print("This bias should be considered when interpreting model results.")

=== Language Distribution (Top 20) ===
+---------------+------+---------+--------+
|primaryLanguage| count|avgRating|avgVotes|
+---------------+------+---------+--------+
|             en|135859|     6.09| 10555.0|
|           NULL| 26623|     6.04|   381.0|
|             ja|  5254|     6.21|   769.0|
|             fr|  4121|     5.84|   962.0|
|             bg|  2680|     6.01|  1253.0|
|             ru|  2616|     6.61|   871.0|
|            cmn|  2179|     6.29|  2363.0|
|             tr|  2176|     5.71|  1169.0|
|             sv|   759|     6.46|   787.0|
|             hi|   596|     6.19|   926.0|
|             ca|   506|     5.88|   911.0|
|             sr|   458|     6.49|   472.0|
|             fa|   403|     5.47|   835.0|
|            qbn|   288|      6.1|   429.0|
|             es|   186|     6.02|  1075.0|
|             he|   168|     6.59|   758.0|
|            yue|   158|     6.12|   540.0|
|             nl|   128|     6.15|   603.0|
|             de|   126|     6.31|   

## 5. numVotes Analysis (Instructor Feedback)

In [9]:
# ── numVotes analysis and log transformation ──
print("=== numVotes Statistics ===")
df.select("numVotes").describe().show()

# Log-transform numVotes
df = df.withColumn("logNumVotes", F.log(F.col("numVotes").cast(FloatType()) + 1))

print("\n=== logNumVotes Statistics ===")
df.select("logNumVotes").describe().show()

# Analyze the relationship between numVotes and rating reliability
print("\n=== Rating Stats by numVotes Buckets ===")
df.withColumn(
    "voteBucket",
    F.when(F.col("numVotes") < 500, "100-499")
    .when(F.col("numVotes") < 1000, "500-999")
    .when(F.col("numVotes") < 5000, "1K-5K")
    .when(F.col("numVotes") < 50000, "5K-50K")
    .otherwise("50K+")
).groupBy("voteBucket").agg(
    F.count("*").alias("count"),
    F.round(F.mean("averageRating"), 2).alias("avgRating"),
    F.round(F.stddev("averageRating"), 2).alias("stdRating")
).orderBy("voteBucket").show()

=== numVotes Statistics ===
+-------+------------------+
|summary|          numVotes|
+-------+------------------+
|  count|            186102|
|   mean| 7893.845858722636|
| stddev|53748.607404259696|
|    min|               100|
|    max|           3183892|
+-------+------------------+


=== logNumVotes Statistics ===
+-------+------------------+
|summary|       logNumVotes|
+-------+------------------+
|  count|            186102|
|   mean| 6.580549329892262|
| stddev|1.6652500063957527|
|    min|  4.61512051684126|
|    max|14.973615219854095|
+-------+------------------+


=== Rating Stats by numVotes Buckets ===
+----------+-----+---------+---------+
|voteBucket|count|avgRating|stdRating|
+----------+-----+---------+---------+
|   100-499|96222|     5.92|     1.31|
|     1K-5K|38653|     6.23|     1.24|
|   500-999|27774|     5.97|     1.31|
|      50K+| 5250|     6.96|     0.98|
|    5K-50K|18203|     6.59|     1.12|
+----------+-----+---------+---------+



## 6. Director & Cast Track Record Features

In [10]:
# ── Director average rating (track record) ──
# Explode directors (comma-separated)
df_dir_ratings = (
    df.select("tconst", "directors", "averageRating")
    .filter(F.col("directors").isNotNull())
    .withColumn("director", F.explode(F.split(F.col("directors"), ",")))
    .groupBy("director")
    .agg(
        F.mean("averageRating").alias("directorAvgRating"),
        F.count("*").alias("directorTitleCount")
    )
)

print(f"Computed track records for {df_dir_ratings.count():,} directors")
df_dir_ratings.orderBy("directorTitleCount", ascending=False).show(10)

Computed track records for 100,073 directors
+---------+------------------+------------------+
| director| directorAvgRating|directorTitleCount|
+---------+------------------+------------------+
|nm0001238| 4.627868852459017|               122|
|nm0437356|6.0636363636363635|               110|
|nm0213983| 4.003636363636364|               110|
|nm0782947| 6.160550458715596|               109|
|nm0676248| 4.258878504672897|               107|
|nm0779641|  7.03551401869159|               107|
|nm0068691|  6.95096153846154|               104|
|nm0002031| 6.509708737864078|               103|
|nm0444709| 7.330097087378641|               103|
|nm0064415| 5.991262135922329|               103|
+---------+------------------+------------------+
only showing top 10 rows


In [11]:
# ── Join director features back (use first director) ──
df = df.withColumn(
    "primaryDirector",
    F.split(F.col("directors"), ",").getItem(0)
)

df = df.join(
    df_dir_ratings.withColumnRenamed("director", "primaryDirector"),
    on="primaryDirector",
    how="left"
)

# Fill missing director features
median_dir_rating = df.filter(F.col("directorAvgRating").isNotNull()).approxQuantile("directorAvgRating", [0.5], 0.01)[0]
df = df.fillna({"directorAvgRating": median_dir_rating, "directorTitleCount": 0})

print(f"Director features added. Median director rating: {median_dir_rating:.2f}")

Director features added. Median director rating: 6.35


In [12]:
# ── Cast quality score ──
# Load principals to get top-billed actors per title
df_principals = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("nullValue", "\\N")
    .option("quote", "")
    .csv(os.path.join(DATA_DIR, "title.principals.tsv.gz"))
)

# Get actors/actresses only (top 3 per title by ordering)
df_actors = (
    df_principals
    .filter(F.col("category").isin(["actor", "actress"]))
    .withColumn("ordering", F.col("ordering").cast(IntegerType()))
    .filter(F.col("ordering") <= 3)
)

# Compute each actor's average rating across titles they appear in
df_actor_ratings = (
    df_actors
    .join(df.select("tconst", "averageRating"), on="tconst", how="inner")
    .groupBy("nconst")
    .agg(F.mean("averageRating").alias("actorAvgRating"))
)

# Compute average cast quality per title
df_cast_quality = (
    df_actors
    .join(df_actor_ratings, on="nconst", how="left")
    .groupBy("tconst")
    .agg(F.mean("actorAvgRating").alias("castAvgRating"))
)

df = df.join(df_cast_quality, on="tconst", how="left")

median_cast = df.filter(F.col("castAvgRating").isNotNull()).approxQuantile("castAvgRating", [0.5], 0.01)[0]
df = df.fillna({"castAvgRating": median_cast})

print(f"Cast quality features added. Median cast rating: {median_cast:.2f}")

Cast quality features added. Median cast rating: 6.13


## 7. Create Rating Tiers (Classification Target)

In [13]:
# ── Rating tiers for classification ──
df = df.withColumn(
    "ratingTier",
    F.when(F.col("averageRating") < 5.0, 0)      # Low
    .when(F.col("averageRating") < 7.0, 1)        # Medium
    .otherwise(2)                                   # High
)

df = df.withColumn(
    "ratingTierLabel",
    F.when(F.col("ratingTier") == 0, "Low (<5)")
    .when(F.col("ratingTier") == 1, "Medium (5-7)")
    .otherwise("High (≥7)")
)

print("Rating tier distribution:")
df.groupBy("ratingTierLabel", "ratingTier").count().orderBy("ratingTier").show()

Rating tier distribution:
+---------------+----------+------+
|ratingTierLabel|ratingTier| count|
+---------------+----------+------+
|       Low (<5)|         0| 33605|
|   Medium (5-7)|         1|103414|
|      High (≥7)|         2| 49083|
+---------------+----------+------+



## 8. Final Feature Matrix

In [14]:
# ── Select final features ──
# Numeric features
numeric_features = [
    "runtimeMinutes", "startYear", "decade",
    "numVotes", "logNumVotes",
    "numRegions", "numPrincipals", "numRoleTypes",
    "isEnglish",
    "directorAvgRating", "directorTitleCount",
    "castAvgRating",
    # User rating features
    "userRatingCount", "userRatingMean", "userRatingStd",
    "extremeRatingRatio", "userRatingRange",
]

# Runtime bucket columns
runtime_cols = ["runtime_short", "runtime_standard", "runtime_long", "runtime_very_long"]

# All feature columns
all_feature_cols = numeric_features + genre_cols + type_cols + runtime_cols

# Targets
target_cols = ["averageRating", "ratingTier", "ratingTierLabel"]

# Metadata (keep for reference, not for modeling)
meta_cols = ["tconst", "primaryTitle", "titleType", "genres", "primaryLanguage"]

print(f"Total feature columns: {len(all_feature_cols)}")
print(f"  Numeric: {len(numeric_features)}")
print(f"  Genre: {len(genre_cols)}")
print(f"  Type: {len(type_cols)}")
print(f"  Runtime: {len(runtime_cols)}")

Total feature columns: 52
  Numeric: 17
  Genre: 27
  Type: 4
  Runtime: 4


In [15]:
# ── Fill remaining nulls ──
df_final = df.select(meta_cols + all_feature_cols + target_cols)
df_final = df_final.fillna(0)

print(f"Final feature matrix: {df_final.count():,} rows × {len(df_final.columns)} columns")
df_final.describe().show()

Final feature matrix: 186,102 rows × 60 columns
+-------+---------+--------------------+---------+-------+---------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-----------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------------+--------------------+-------------------+--------------------+--------------------+-------------------+-------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------------+--------------------+--------

In [16]:
# ── Save feature matrix ──
feature_path = os.path.join(OUTPUT_DIR, "parquet", "features")
df_final.write.mode("overwrite").parquet(feature_path)

print(f"Feature matrix saved to: {feature_path}")

# Also save feature column names for later use
import json
feature_meta = {
    "numeric_features": numeric_features,
    "genre_cols": genre_cols,
    "type_cols": type_cols,
    "runtime_cols": runtime_cols,
    "all_feature_cols": all_feature_cols,
    "target_cols": target_cols,
    "meta_cols": meta_cols
}
with open(os.path.join(OUTPUT_DIR, "feature_columns.json"), "w") as f:
    json.dump(feature_meta, f, indent=2)

print("Feature metadata saved.")
print("\nDone! Proceed to 03_visualization.ipynb")

Feature matrix saved to: e:\CUFE\Spring_25\Big Data\Project\outputs\parquet\features
Feature metadata saved.

Done! Proceed to 03_visualization.ipynb
